Visualizes fluorescence intensities of two channels and their ratios

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import skimage.io as io
from skimage import exposure
from skimage.filters import (
    gaussian,
    threshold_otsu,
    threshold_multiotsu,
    unsharp_mask,
)
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, binary_dilation, disk
from skimage.segmentation import find_boundaries

from cellpose import models

import czifile

model = models.Cellpose(model_type='cyto3')

In [2]:
MIN_INCLUSION_SIZE = 10
MAX_INCLUSION_SIZE = 10000
COLOR_SCHEME = 'viridis'

In [ ]:
def show_image(image, cmap=COLOR_SCHEME):
    plt.imshow(image, cmap=cmap)
    plt.colorbar()
    plt.show()


def show_ratio_image(image, cmap=COLOR_SCHEME):
    plt.imshow(image, cmap=cmap, vmin=0, vmax=5)
    plt.colorbar()
    plt.show()


def show_images(images, titles=None, cmap=COLOR_SCHEME):
    n = len(images)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))

    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for i, img in enumerate(images):
        ax = axes[i // cols][i % cols]
        im = ax.imshow(img, cmap=cmap)
        if titles:
            ax.set_title(titles[i])
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for j in range(i + 1, rows * cols):
        ax = axes[j // cols][j % cols]
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_image(image_path, basename, verbose=False):
    """
    Analyze an image by its individual cells.
    """
    image = czifile.imread(image_path)
    image_squeezed = np.squeeze(image)

    orange_channel = gaussian(image_squeezed[0], sigma=2)
    green_channel = gaussian(image_squeezed[1], sigma=2)

    ratio_image = np.nan_to_num(orange_channel / green_channel, nan=0, posinf=0, neginf=0)

    show_images(
        [orange_channel, green_channel, ratio_image],
        titles=['Orange Channel', 'Green Channel', 'Ratio Image']
    )


In [ ]:
def analyze_all_images(image_folder, images=None, verbose=False):
    """
    Analyze all .czi images in a folder.
    """
    print("Analyzing images in folder:", image_folder)

    for filename in os.listdir(image_folder):
        if not filename.lower().endswith(".czi"):
            continue
        if images and filename not in images:
            continue

        image_path = os.path.join(image_folder, filename)
        base_name = os.path.splitext(filename)[0]

        if verbose:
            print("-" * 100)
            print("Image:", image_path)

        analyze_image(image_path, base_name, verbose=verbose)

        if verbose:
            print("-" * 100)


In [ ]:
folders = ['61025_FRET','27525_FRET','060325_FRET','51325FRET','052025_FRET']
for folder in folders:
    analyze_all_images(folder,verbose=True)